# Multi-Factor Strategy: RSI + MACD + Bollinger Bands

This notebook demonstrates how to build a multi-factor trading strategy that combines multiple technical indicators for more robust signals.

## Strategy Overview

The **Multi-Factor Strategy** combines three indicators:

1. **RSI (Relative Strength Index)** - Momentum indicator
2. **MACD (Moving Average Convergence Divergence)** - Trend indicator
3. **Bollinger Bands** - Volatility indicator

Buy signal requires ALL conditions:
- RSI < 30 (oversold)
- MACD histogram turns positive
- Price below lower Bollinger Band

Sell signal requires ALL conditions:
- RSI > 70 (overbought)
- MACD histogram turns negative
- Price above upper Bollinger Band

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

from cbsc_strategy_sdk import StrategyWorkspace, BacktestAdapter

print("✅ Imports complete")

## Define Indicator Functions

In [ ]:
def calculate_rsi(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Calculate RSI."""
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(df: pd.DataFrame, fast: int = 12, slow: int = 26, signal: int = 9):
    """Calculate MACD."""
    ema_fast = df['close'].ewm(span=fast).mean()
    ema_slow = df['close'].ewm(span=slow).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def calculate_bollinger_bands(df: pd.DataFrame, period: int = 20, std_dev: float = 2.0):
    """Calculate Bollinger Bands."""
    middle_band = df['close'].rolling(window=period).mean()
    std = df['close'].rolling(window=period).std()
    upper_band = middle_band + (std_dev * std)
    lower_band = middle_band - (std_dev * std)
    return upper_band, middle_band, lower_band

print("✅ Indicator functions defined")

## Fetch and Prepare Data

In [ ]:
async def prepare_data():
    async with StrategyWorkspace() as ws:
        data = await ws.get_historical_data(
            symbol="AAPL",
            start=date(2023, 1, 1),
            end=date(2024, 12, 31)
        )
    return data

data = await prepare_data()

# Calculate all indicators
data['rsi'] = calculate_rsi(data)
data['macd'], data['macd_signal'], data['macd_hist'] = calculate_macd(data)
data['bb_upper'], data['bb_middle'], data['bb_lower'] = calculate_bollinger_bands(data)

print("📊 Data prepared with indicators")
print(data[['close', 'rsi', 'macd_hist', 'bb_upper', 'bb_lower']].tail())

## Generate Multi-Factor Signals

In [ ]:
# Generate signals based on multiple conditions
data['buy_condition'] = (
    (data['rsi'] < 30) &  # Oversold
    (data['macd_hist'] > 0) &  # MACD positive
    (data['close'] < data['bb_lower'])  # Below lower band
)

data['sell_condition'] = (
    (data['rsi'] > 70) &  # Overbought
    (data['macd_hist'] < 0) &  # MACD negative
    (data['close'] > data['bb_upper'])  # Above upper band
)

# Count signals
buy_count = data['buy_condition'].sum()
sell_count = data['sell_condition'].sum()

print(f"📊 Signal Counts:")
print(f"  Buy signals: {buy_count}")
print(f"  Sell signals: {sell_count}")

## Visualize Multi-Factor Strategy

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price and Bollinger Bands
axes[0].plot(data.index, data['close'], label='Price', alpha=0.7)
axes[0].plot(data.index, data['bb_upper'], 'r--', label='Upper BB', alpha=0.5)
axes[0].plot(data.index, data['bb_lower'], 'r--', label='Lower BB', alpha=0.5)
axes[0].scatter(data[data['buy_condition']].index, data.loc[data['buy_condition'], 'close'],
              color='green', marker='^', s=100, label='Buy', zorder=5)
axes[0].scatter(data[data['sell_condition']].index, data.loc[data['sell_condition'], 'close'],
              color='red', marker='v', s=100, label='Sell', zorder=5)
axes[0].set_title('Multi-Factor Strategy: Price & Bollinger Bands')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(data.index, data['rsi'], label='RSI', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5)
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5)
axes[1].fill_between(data.index, 30, 70, alpha=0.1)
axes[1].set_title('RSI (Overbought > 70, Oversold < 30)')
axes[1].set_ylabel('RSI')
axes[1].grid(True, alpha=0.3)

# MACD Histogram
colors = ['g' if x > 0 else 'r' for x in data['macd_hist']]
axes[2].bar(data.index, data['macd_hist'], color=colors, alpha=0.6)
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_title('MACD Histogram')
axes[2].set_ylabel('Histogram')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Multi-factor visualization created")

## Run Backtest

In [ ]:
strategy_code = """
def generate_signals(df):
    # Calculate indicators
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    
    ema_fast = df['close'].ewm(span=12).mean()
    ema_slow = df['close'].ewm(span=26).mean()
    macd_hist = (ema_fast - ema_slow) - (ema_fast - ema_slow).ewm(span=9).mean()
    
    bb_middle = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    bb_upper = bb_middle + 2 * bb_std
    bb_lower = bb_middle - 2 * bb_std
    
    # Generate signals
    signals = pd.DataFrame(index=df.index)
    signals['signal'] = 0
    
    # Buy: All three conditions met
    buy_mask = (rsi < 30) & (macd_hist > 0) & (df['close'] < bb_lower)
    signals.loc[buy_mask, 'signal'] = 1
    
    # Sell: All three conditions met
    sell_mask = (rsi > 70) & (macd_hist < 0) & (df['close'] > bb_upper)
    signals.loc[sell_mask, 'signal'] = -1
    
    return signals
"""

async def run_backtest():
    async with BacktestAdapter() as adapter:
        result = await adapter.run_backtest(
            strategy_code=strategy_code,
            symbols=["AAPL"],
            start_date=date(2023, 1, 1),
            end_date=date(2024, 12, 31)
        )
    return result

result = await run_backtest()

print("✅ Backtest complete")
print(f"\n📊 Performance:")
print(f"  Total Return: {result.metrics.total_return:.2%}")
print(f"  Sharpe Ratio: {result.metrics.sharpe_ratio:.2f}")
print(f"  Max Drawdown: {result.metrics.max_drawdown:.2%}")
print(f"  Win Rate: {result.metrics.win_rate:.2%}")

## Summary

Multi-factor strategies combine multiple indicators to:

✅ **Reduce false signals** - Require confirmation from multiple sources  
✅ **Improve robustness** - Work across different market conditions  
✅ **Enhance risk-adjusted returns** - Better Sharpe ratios  

## Key Takeaways

1. Combine indicators from different categories (momentum, trend, volatility)
2. Require ALL conditions for entry (confluence)
3. Use ANY condition for exit (flexible risk management)
4. Backtest thoroughly to validate the combination

---
**Version:** 0.1.0